In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import resnet50
from tensorflow.keras.preprocessing import image_dataset_from_directory
import numpy as np
from pathlib import Path
import h5py, sys, os
DATA_DIR = Path(r"C:\Users\hugop\OneDrive\Bureau\ML_project\data") 
IMG_SIZE = (224, 224) 
SEED = 42
BATCH = 32  
BATCH = 32
train_ds = image_dataset_from_directory(
    DATA_DIR, labels='inferred', label_mode='int',
    validation_split=0.2, subset='training', seed=SEED,
    image_size=IMG_SIZE, batch_size=BATCH, shuffle=True
)
val_ds = image_dataset_from_directory(
    DATA_DIR, labels='inferred', label_mode='int',
    validation_split=0.2, subset='validation', seed=SEED,
    image_size=IMG_SIZE, batch_size=BATCH, shuffle=False
)
class_names = train_ds.class_names
base = resnet50.ResNet50(include_top=False, pooling="avg",
                         weights="imagenet", input_shape=IMG_SIZE+(3,))
preprocess = resnet50.preprocess_input
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE); val_ds = val_ds.prefetch(AUTOTUNE)
def ds_to_features(ds):
    Xf, yf = [], []
    for imgs, labels in ds:
        feats = base(preprocess(imgs))
        Xf.append(feats.numpy()); yf.append(labels.numpy())
    return np.concatenate(Xf), np.concatenate(yf)
X_tr_deep, y_tr_deep = ds_to_features(train_ds)
X_val_deep, y_val_deep = ds_to_features(val_ds)
X_tr_deep.shape, X_val_deep.shape, len(class_names)


Found 2527 files belonging to 6 classes.
Using 2022 files for training.
Found 2527 files belonging to 6 classes.
Using 505 files for validation.


((2022, 2048), (505, 2048), 6)

In [ ]:

from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
from pathlib import Path
import joblib
import numpy as np
import gc
try:
    import tensorflow as tf
    tf.keras.backend.clear_session()
except Exception:
    pass
gc.collect()
X_tr_deep = X_tr_deep.astype(np.float32, copy=False)
X_val_deep = X_val_deep.astype(np.float32, copy=False)
rf = RandomForestClassifier(n_estimators=600, random_state=SEED, n_jobs=-1)
rf.fit(X_tr_deep, y_tr_deep)
pred = rf.predict(X_val_deep)
print("ResNet/VIT + RF  | Acc:",
      accuracy_score(y_val_deep, pred), "| F1-macro:", f1_score(y_val_deep, pred, average="macro"))
print(classification_report(y_val_deep, pred, target_names=class_names, digits=3))
Path("models/deep").mkdir(parents=True, exist_ok=True)
joblib.dump(rf, Path("models/deep") / "resnet_rf.joblib")
hgb = HistGradientBoostingClassifier(random_state=SEED, max_iter=300)
hgb.fit(X_tr_deep, y_tr_deep)
pred = hgb.predict(X_val_deep)
print("ResNet/VIT + HistGB | Acc:",
      accuracy_score(y_val_deep, pred), "| F1-macro:", f1_score(y_val_deep, pred, average="macro"))
print(classification_report(y_val_deep, pred, target_names=class_names, digits=3))
joblib.dump(hgb, Path("models/deep") / "resnet_histgb.joblib")



ResNet/VIT + RF  | Acc: 0.9168316831683169 | F1-macro: 0.31030040448704027
              precision    recall  f1-score   support

   cardboard      0.000     0.000     0.000         0
       glass      0.000     0.000     0.000         0
       metal      0.000     0.000     0.000         0
       paper      0.000     0.000     0.000         0
     plastic      0.983     0.959     0.971       368
       trash      1.000     0.803     0.891       137

    accuracy                          0.917       505
   macro avg      0.331     0.294     0.310       505
weighted avg      0.988     0.917     0.949       505



c:\Users\hugop\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\hugop\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\hugop\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


ResNet/VIT + HistGB | Acc: 0.9425742574257425 | F1-macro: 0.3202153279501346
              precision    recall  f1-score   support

   cardboard      0.000     0.000     0.000         0
       glass      0.000     0.000     0.000         0
       metal      0.000     0.000     0.000         0
       paper      0.000     0.000     0.000         0
     plastic      0.992     0.959     0.975       368
       trash      1.000     0.898     0.946       137

    accuracy                          0.943       505
   macro avg      0.332     0.310     0.320       505
weighted avg      0.994     0.943     0.967       505



c:\Users\hugop\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\hugop\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\hugop\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


['models\\deep\\resnet_histgb.joblib']

In [3]:
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
import joblib, numpy as np
from pathlib import Path
rf = RandomForestClassifier(n_estimators=600, random_state=SEED, n_jobs=-1)
rf.fit(X_tr_deep, y_tr_deep)
pred = rf.predict(X_val_deep)
print("ResNet + RF  | Acc:",
      accuracy_score(y_val_deep, pred), "| F1-macro:", f1_score(y_val_deep, pred, average="macro"))
print(classification_report(y_val_deep, pred, target_names=class_names, digits=3))
Path("models/deep").mkdir(parents=True, exist_ok=True)
joblib.dump(rf, Path("models/deep") / "resnet_rf.joblib")
hgb = HistGradientBoostingClassifier(random_state=SEED, max_iter=300)
hgb.fit(X_tr_deep, y_tr_deep)
pred = hgb.predict(X_val_deep)
print("ResNet + HistGB | Acc:",
      accuracy_score(y_val_deep, pred), "| F1-macro:", f1_score(y_val_deep, pred, average="macro"))
print(classification_report(y_val_deep, pred, target_names=class_names, digits=3))
joblib.dump(hgb, Path("models/deep") / "resnet_histgb.joblib")


ResNet + RF  | Acc: 0.9168316831683169 | F1-macro: 0.31030040448704027
              precision    recall  f1-score   support

   cardboard      0.000     0.000     0.000         0
       glass      0.000     0.000     0.000         0
       metal      0.000     0.000     0.000         0
       paper      0.000     0.000     0.000         0
     plastic      0.983     0.959     0.971       368
       trash      1.000     0.803     0.891       137

    accuracy                          0.917       505
   macro avg      0.331     0.294     0.310       505
weighted avg      0.988     0.917     0.949       505



c:\Users\hugop\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\hugop\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\hugop\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


ResNet + HistGB | Acc: 0.9425742574257425 | F1-macro: 0.3202153279501346
              precision    recall  f1-score   support

   cardboard      0.000     0.000     0.000         0
       glass      0.000     0.000     0.000         0
       metal      0.000     0.000     0.000         0
       paper      0.000     0.000     0.000         0
     plastic      0.992     0.959     0.975       368
       trash      1.000     0.898     0.946       137

    accuracy                          0.943       505
   macro avg      0.332     0.310     0.320       505
weighted avg      0.994     0.943     0.967       505



c:\Users\hugop\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\hugop\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\hugop\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


['models\\deep\\resnet_histgb.joblib']

In [ ]:

import timm, torch, gc, random, json
from torchvision import transforms
from PIL import Image
import numpy as np
from pathlib import Path
DATA_DIR = Path(r"C:\Users\hugop\OneDrive\Bureau\ML_project\data") 
IMG_SIZE = (224, 224)
SEED = 42

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_num_threads(1)  # évite la sursaturation du cpu (ça depend des ordis)
vit = timm.create_model('vit_small_patch16_224', pretrained=True, num_classes=0).to(device).eval()
tfm = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225))
])
labels_json_path = DATA_DIR.parent / "labels.json"
if labels_json_path.exists():
    with open(labels_json_path, "r", encoding="utf-8") as f:
        class_to_idx = json.load(f)
    # assure un ordre par index
    class_names = [c for c, _ in sorted(class_to_idx.items(), key=lambda x: x[1])]
else:
    class_names = sorted([d.name for d in DATA_DIR.iterdir() if d.is_dir()])
    class_to_idx = {c: i for i, c in enumerate(class_names)}
    with open(labels_json_path, "w", encoding="utf-8") as f:
        json.dump(class_to_idx, f, ensure_ascii=False, indent=2)
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
paths, labels = [], []
for c in class_names:
    for p in (DATA_DIR / c).glob("*"):
        if p.suffix.lower() in IMG_EXTS:
            paths.append(p)
            labels.append(class_to_idx[c])
idx = list(range(len(paths)))
random.seed(SEED); random.shuffle(idx)
split = int(0.8 * len(idx))
tr_idx, va_idx = idx[:split], idx[split:]
@torch.no_grad()
def extract_feats(idxs, chunk=32):
    X, y = [], []
    for s in range(0, len(idxs), chunk):
        batch = idxs[s:s+chunk]
        imgs, ys = [], []
        for i in batch:
            with Image.open(paths[i]).convert("RGB") as im:
                imgs.append(tfm(im).unsqueeze(0))
            ys.append(labels[i])
        t = torch.cat(imgs, dim=0).to(device, non_blocking=True)
        feats = vit(t).detach().cpu().numpy().astype(np.float32)
        X.append(feats)
        y.extend(ys)
        del t, imgs, feats
        if device == "cuda":
            torch.cuda.empty_cache()
        gc.collect()
    return np.vstack(X), np.array(y, dtype=np.int32)
X_tr_vit, y_tr_vit = extract_feats(tr_idx, chunk=32)
X_val_vit, y_val_vit = extract_feats(va_idx, chunk=32)
print(X_tr_vit.shape, X_val_vit.shape)


(2021, 384) (506, 384)


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
import joblib, numpy as np
from pathlib import Path
X_tr = X_tr_vit.astype(np.float32, copy=False)
X_val = X_val_vit.astype(np.float32, copy=False)
y_tr = y_tr_vit
y_val = y_val_vit
n_comp = int(min(256, X_tr.shape[1], max(1, X_tr.shape[0]-1)))
rf_pipeline = Pipeline([
    ("pca", PCA(n_components=n_comp, random_state=SEED)),
    ("rf", RandomForestClassifier(
        n_estimators=300, max_features="sqrt",
        random_state=SEED, n_jobs=-1
    ))
])
hgb_pipeline = Pipeline([
    ("pca", PCA(n_components=n_comp, random_state=SEED)),
    ("clf", HistGradientBoostingClassifier(random_state=SEED, max_iter=300))
])
def fit_eval(name, pipe):
    pipe.fit(X_tr, y_tr)
    pred = pipe.predict(X_val)
    print(f"{name} | Acc: {accuracy_score(y_val, pred):.4f} | F1-macro: {f1_score(y_val, pred, average='macro'):.4f}")
    print(classification_report(y_val, pred, target_names=class_names, digits=3))
    return pipe
rf_trained  = fit_eval("ViT + PCA + RandomForest", rf_pipeline)
hgb_trained = fit_eval("ViT + PCA + HistGB",      hgb_pipeline)
models_dir = Path(r"C:\Users\hugop\OneDrive\Bureau\ML_project\models\deep")
models_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(rf_trained,  models_dir / f"vit_pca{n_comp}_rf.joblib")
joblib.dump(hgb_trained, models_dir / f"vit_pca{n_comp}_histgb.joblib")


ViT + PCA + RandomForest | Acc: 0.8597 | F1-macro: 0.7595
              precision    recall  f1-score   support

   cardboard      0.987     0.881     0.931        84
       glass      0.896     0.878     0.887        98
       metal      0.892     0.892     0.892        93
       paper      0.776     0.983     0.867       116
     plastic      0.826     0.826     0.826        92
       trash      0.667     0.087     0.154        23

    accuracy                          0.860       506
   macro avg      0.841     0.758     0.759       506
weighted avg      0.860     0.860     0.846       506

ViT + PCA + HistGB | Acc: 0.8794 | F1-macro: 0.8303
              precision    recall  f1-score   support

   cardboard      0.987     0.929     0.957        84
       glass      0.896     0.878     0.887        98
       metal      0.876     0.914     0.895        93
       paper      0.836     0.966     0.896       116
     plastic      0.852     0.815     0.833        92
       trash      0.75

['C:\\Users\\hugop\\OneDrive\\Bureau\\ML_project\\models\\deep\\vit_pca256_histgb.joblib']